# Frame-Based Knowledge Representation

## Introduction
Frame-based knowledge representation organizes information about objects or concepts into structured units called **frames**. Each frame contains named **slots**, and each slot stores a property or value. Frames can be arranged in a hierarchy so that a specialized frame inherits information from a more general parent frame.

A child frame can also define its own value for a slot. This local value takes precedence over the inherited value. This is useful for representing exceptions, such as a penguin inheriting general bird properties while having `canFly = False`.

## Aim
To implement a simple frame-based knowledge representation system in Python using frames, slots, inheritance, and slot overriding.

## Learning objectives
- Understand frames and slots.
- Represent a hierarchy of related concepts.
- Retrieve slot values through inheritance.
- Override inherited values in a child frame.
- Add new frames dynamically.
- Validate frame references and detect circular inheritance.

## 1. Theory: Frames and Slots

A **frame** is a structured representation of an entity, object, concept, or stereotyped situation. A frame groups related knowledge in one place.

A **slot** is a named attribute in a frame. For example, an `Animal` frame may contain slots such as `canMove`, `breathes`, and `hasCells`. Slot values may be Boolean, numeric, textual, or references to other frames.

### Inheritance
A child frame can inherit slot values from its parent. If the child does not define a particular slot, the system searches the parent frame and continues upward through the hierarchy.

### Slot overriding
If a child frame defines a slot that also exists in its parent, the child's local value is returned. This lets the knowledge base represent exceptions to general or default information.

### Missing values
If a slot is not defined in the frame or any ancestor, this implementation returns `None`, meaning that the value is not available in the knowledge base.

## 2. Create the Frame Knowledge Base

The example uses an `Animal → Bird → Penguin` hierarchy. `Bird` inherits general animal properties, while `Penguin` inherits bird properties and overrides the value of `canFly`.

In [ ]:
frames = {
    "Animal": {
        "canMove": True,
        "breathes": True,
        "hasCells": True
    },
    "Bird": {
        "parent": "Animal",
        "hasWings": True,
        "canFly": True
    },
    "Penguin": {
        "parent": "Bird",
        "canFly": False,
        "canSwim": True
    }
}

print("Frames created successfully.")
print("Available frames:", list(frames.keys()))

## 3. Retrieve Slot Values Using Inheritance

The `get_slot()` function first checks the requested frame. If the slot is not present locally, it follows the `parent` reference. The first matching slot is returned, so a child frame's value has priority over a parent's value.

The function also checks whether a frame exists and tracks visited frames to prevent infinite recursion if a circular parent relationship is present.

In [ ]:
def get_slot(frame_name, slot, visited=None):
    """Return a slot value, searching the frame and then its ancestors."""
    if frame_name not in frames:
        raise KeyError(f"Frame '{frame_name}' does not exist.")

    if visited is None:
        visited = set()

    if frame_name in visited:
        raise ValueError(f"Circular inheritance detected at frame '{frame_name}'.")

    visited.add(frame_name)
    frame = frames[frame_name]

    # A locally defined slot takes precedence over inherited values.
    if slot in frame:
        return frame[slot]

    parent = frame.get("parent")
    if parent is None:
        return None

    return get_slot(parent, slot, visited)


print("Penguin can move:", get_slot("Penguin", "canMove"))
print("Penguin can fly:", get_slot("Penguin", "canFly"))
print("Penguin can swim:", get_slot("Penguin", "canSwim"))
print("Penguin breathes:", get_slot("Penguin", "breathes"))

## 4. Demonstrate Inheritance and Slot Overriding

The `Bird` frame defines `canFly = True`. The `Penguin` frame defines `canFly = False`. Since the local slot is checked first, the penguin's value overrides the inherited bird value.

This is an example of exception handling in a frame hierarchy: a general property is inherited unless a more specific frame supplies another value.

In [ ]:
print("Bird can fly:", get_slot("Bird", "canFly"))
print("Penguin can fly:", get_slot("Penguin", "canFly"))

assert get_slot("Bird", "canFly") is True
assert get_slot("Penguin", "canFly") is False
print("Slot overriding verified.")

## 5. Add a New Frame Dynamically

A new frame can be added after the initial knowledge base has been created. The `add_frame()` function checks that the frame name is unique and that its parent already exists.

The `Eagle` frame inherits general properties from `Bird` and `Animal`, while defining its own values for `canFly` and `hasSharpVision`.

In [ ]:
def add_frame(name, parent=None, slots=None):
    """Add a frame after validating its name and parent reference."""
    if name in frames:
        raise ValueError(f"Frame '{name}' already exists.")

    if parent is not None and parent not in frames:
        raise KeyError(f"Parent frame '{parent}' does not exist.")

    new_slots = dict(slots or {})
    if "parent" in new_slots:
        raise ValueError("Specify the parent using the parent argument, not inside slots.")

    if parent is not None:
        new_slots["parent"] = parent

    frames[name] = new_slots


add_frame(
    "Eagle",
    parent="Bird",
    slots={"canFly": True, "hasSharpVision": True}
)

print("Eagle can move:", get_slot("Eagle", "canMove"))
print("Eagle can fly:", get_slot("Eagle", "canFly"))
print("Eagle has sharp vision:", get_slot("Eagle", "hasSharpVision"))

## 6. Handle Missing Slots and Invalid Frames

A query for a slot that is not present anywhere in the inheritance chain returns `None`. A query for a frame that does not exist raises a `KeyError`, because the requested frame cannot be searched.

These behaviours make the result of a query clearer and prevent silent failures caused by misspelled frame names.

In [ ]:
print("Penguin has feathers slot:", get_slot("Penguin", "hasFeathers"))
print("Unknown slot returns None:", get_slot("Penguin", "unknownSlot") is None)

try:
    get_slot("Dragon", "canFly")
except KeyError as error:
    print("Invalid frame handled:", error)

## 7. Validate the Knowledge Base

The following tests verify inherited values, locally defined values, overridden values, dynamically added frames, missing slots, and invalid frame names. Assertions stop execution if an expected result is incorrect.

In [ ]:
# Inheritance from Animal through Bird to Penguin
assert get_slot("Penguin", "canMove") is True
assert get_slot("Penguin", "breathes") is True
assert get_slot("Penguin", "hasCells") is True

# Local values and overriding
assert get_slot("Penguin", "canFly") is False
assert get_slot("Bird", "canFly") is True
assert get_slot("Penguin", "canSwim") is True

# Dynamically added frame inherits and overrides correctly
assert get_slot("Eagle", "canMove") is True
assert get_slot("Eagle", "canFly") is True
assert get_slot("Eagle", "hasSharpVision") is True

# Missing slot
assert get_slot("Penguin", "unknownSlot") is None

print("All inheritance and slot-value tests passed.")

## 8. Detect Circular Inheritance

A valid single-parent hierarchy should not contain a cycle. The lookup function includes cycle detection so it cannot recurse indefinitely. The following test temporarily creates a circular reference, verifies that it is detected, and then restores the knowledge base.

In [ ]:
# Save the original parent reference so the example can be safely restored.
original_animal_parent = frames["Animal"].get("parent")

try:
    frames["Animal"]["parent"] = "Penguin"
    get_slot("Penguin", "slotThatDoesNotExist")
except ValueError as error:
    print("Circular inheritance handled:", error)
finally:
    if original_animal_parent is None:
        frames["Animal"].pop("parent", None)
    else:
        frames["Animal"]["parent"] = original_animal_parent

# Confirm normal lookup still works after restoring the hierarchy.
assert get_slot("Penguin", "canMove") is True
print("Knowledge base restored successfully.")

## 9. Observations

- Frames store knowledge as named slots and their values.
- A child frame inherits values that are not defined locally.
- A locally defined slot takes precedence over the inherited value.
- New frames can extend the knowledge base without rewriting existing frames.
- An unknown slot returns `None`, while an unknown frame raises a `KeyError`.
- Circular inheritance is detected to prevent endless recursive lookup.

## 10. Conclusion

This implementation demonstrates frame-based knowledge representation using Python dictionaries. It represents concepts through frames, stores properties in slots, and supports hierarchical inheritance. Slot overriding allows specific frames to represent exceptions to general properties. Validation and cycle detection make the example more reliable.

The program is an educational single-parent frame system. It does not implement advanced frame-system features such as multiple inheritance, slot facets, or procedural attachments.

## References

- University of Cambridge, *Artificial Intelligence* lecture notes: discussion of frame inheritance, slots, and defaults.
- Frames as a knowledge representation technique, Artificial Intelligence course notes: discussion of slots and default values.

These references provide background on the frame concepts illustrated in this notebook.